# ml_B_lasso_case.ipynb

**所属章节**：Chapter B 惩罚回归  
**课程**：金融数据分析与建模 Part V  
**作者**：连玉君（中山大学岭南学院）  
**最后更新**：2026-04

**案例主题**：比较 OLS、Lasso、Ridge、弹性网、Post-Lasso 五种方法的变量筛选与样本外预测。

**运行说明**：按顺序执行所有 Cell，约 1 分钟。


In [ ]:
# 全局设置
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.linear_model import (
    Lasso, LassoCV, Ridge, RidgeCV, ElasticNet, ElasticNetCV, LinearRegression)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import warnings, os
warnings.filterwarnings('ignore')
os.makedirs('data', exist_ok=True)
os.makedirs('figs', exist_ok=True)

FP  = fm.FontProperties(fname='/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc')
FPB = fm.FontProperties(fname='/usr/share/fonts/opentype/noto/NotoSansCJK-Medium.ttc')
C = {'primary':'#2166AC','secondary':'#D6604D','tertiary':'#4DAC26',
     'neutral':'#878787','highlight':'#B2182B','fill':'#D1E5F0'}
plt.rcParams.update({'figure.dpi':120,'savefig.dpi':300,
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.25,'grid.linestyle':'--'})
SEED = 42
np.random.seed(SEED)
print('Global setup done')

---
## 1. 数据生成与分割

**DGP**：稀疏线性回归，n=300, p=50, s=8（真实非零系数数），AR(1) 相关特征，rho=0.5。


In [ ]:
# 数据生成过程 (DGP)
n, p, s = 300, 50, 8
rho = 0.5
# AR(1) 协方差矩阵
Sigma = rho ** np.abs(np.subtract.outer(np.arange(p), np.arange(p)))
X_raw = np.random.multivariate_normal(np.zeros(p), Sigma, size=n)

# 真实稀疏系数（前 s 个非零）
coef_true = np.zeros(p)
signs = np.random.choice([-1, 1], size=s)
coef_true[:s] = signs * np.random.uniform(1, 3, size=s)

# 目标变量
y = X_raw @ coef_true + np.random.normal(0, 1, n)

# 保存数据
df_save = pd.DataFrame(X_raw, columns=[f'x{j+1}' for j in range(p)])
df_save['y'] = y
df_save.to_csv('data/sim_B_sparse.csv', index=False)

# 训练集 / 测试集分割（70/30）
from sklearn.model_selection import train_test_split
X_tr_raw, X_te_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.3, random_state=SEED)

# 标准化（只在训练集 fit，防止数据泄露）
sc = StandardScaler()
X_train = sc.fit_transform(X_tr_raw)
X_test  = sc.transform(X_te_raw)

true_support = set(np.where(coef_true != 0)[0])
print(f'训练集: {X_train.shape}, 测试集: {X_test.shape}')
print(f'真实非零变量: {sorted(true_support)}')

---
## 2. 五种方法的估计


In [ ]:
# 方法一：OLS（基准）
ols = LinearRegression().fit(X_train, y_train)
yp_ols  = ols.predict(X_test)

# 方法二：Lasso（LassoCV 自动选调节参数）
lasso_cv = LassoCV(cv=10, n_alphas=100, random_state=SEED, max_iter=5000)
lasso_cv.fit(X_train, y_train)
yp_lasso = lasso_cv.predict(X_test)
lasso_sel = set(np.where(lasso_cv.coef_ != 0)[0])

# 方法三：Ridge（RidgeCV 自动选调节参数）
ridge_cv = RidgeCV(alphas=np.logspace(-2, 4, 50), cv=10)
ridge_cv.fit(X_train, y_train)
yp_ridge = ridge_cv.predict(X_test)

# 方法四：弹性网（ElasticNetCV 同时选 lambda 和 alpha）
enet_cv = ElasticNetCV(
    l1_ratio=[0.1,0.3,0.5,0.7,0.9,0.95,1.0],
    cv=10, n_alphas=50, random_state=SEED, max_iter=5000)
enet_cv.fit(X_train, y_train)
yp_enet  = enet_cv.predict(X_test)
enet_sel = set(np.where(enet_cv.coef_ != 0)[0])

# 方法五：Post-Lasso（两步：Lasso筛变量 -> OLS精估计）
if len(lasso_sel) > 0:
    lasso_sel_list = sorted(lasso_sel)
    post_ols = LinearRegression().fit(X_train[:, lasso_sel_list], y_train)
    yp_post  = post_ols.predict(X_test[:, lasso_sel_list])
else:
    yp_post = np.full(len(y_test), y_train.mean())

print(f'Lasso:  lambda*={lasso_cv.alpha_:.4f}, 选中变量={len(lasso_sel)}/{p}')
print(f'Ridge:  lambda*={ridge_cv.alpha_:.4f}')
print(f'弹性网: lambda*={enet_cv.alpha_:.4f}, alpha={enet_cv.l1_ratio_:.2f}, 选中={len(enet_sel)}/{p}')

---
## 3. 样本外评估


In [ ]:
# 样本外 R² 计算函数
def oos_r2(y_true, y_pred, baseline_mean):
    # 样本外 R2 = 1 - MSE(model) / MSE(baseline)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - baseline_mean) ** 2)
    return 1 - ss_res / ss_tot

bm = y_train.mean()  # 基准：训练集均值
results = {}
for name, yp in [('OLS',yp_ols),('Lasso',yp_lasso),
                  ('Ridge',yp_ridge),('弹性网',yp_enet),('Post-Lasso',yp_post)]:
    mse = mean_squared_error(y_test, yp)
    results[name] = {
        'MSE' : round(mse, 4),
        'RMSE': round(np.sqrt(mse), 4),
        'OOS_R2': round(oos_r2(y_test, yp, bm), 4)
    }

df_res = pd.DataFrame(results).T
df_res.index.name = 'Method'
print('=' * 50)
print('样本外预测评估结果（测试集）')
print('=' * 50)
print(df_res.to_string())

---
## 4. 变量筛选准确率


In [ ]:
# 计算精确率、召回率、F1
for method_name, selected in [('Lasso', lasso_sel), ('弹性网', enet_sel)]:
    if not selected:
        print(f'{method_name}: 未选中任何变量'); continue
    tp = len(true_support & selected)
    fp = len(selected - true_support)
    fn = len(true_support - selected)
    prec   = tp / len(selected)
    recall = tp / len(true_support)
    f1     = 2*prec*recall/(prec+recall) if prec+recall > 0 else 0
    print(f'{method_name} (选中{len(selected)}个):')
    print(f'  精确率={prec:.3f}, 召回率={recall:.3f}, F1={f1:.3f}')
    print(f'  正确选入: {sorted(true_support & selected)}')
    print(f'  遗漏: {sorted(true_support - selected)}')
    print(f'  误选: {sorted(selected - true_support)}')
    print()

---
## 5. 结果可视化


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.subplots_adjust(wspace=0.35)

# 左图：预测值 vs 真实值
ax = axes[0]
for yp, label, col, sz in [
    (yp_ols, 'OLS', C['neutral'], 18),
    (yp_lasso, 'Lasso', C['primary'], 22),
    (yp_post, 'Post-Lasso', C['secondary'], 22)
]:
    ax.scatter(y_test, yp, color=col, alpha=0.55, s=sz, label=label)
lim = max(abs(y_test).max(), abs(yp_lasso).max()) * 1.1
ax.plot([-lim,lim],[-lim,lim], 'k--', lw=1, alpha=0.4)
ax.set_xlabel('真实值 y', fontproperties=FP, fontsize=11)
ax.set_ylabel('预测值', fontproperties=FP, fontsize=11)
ax.set_title('预测值 vs 真实值', fontproperties=FPB, fontsize=12)
ax.legend(prop=FP)

# 右图：OOS R2 对比条形图
ax2 = axes[1]
methods  = df_res.index.tolist()
r2_vals  = df_res['OOS_R2'].values
cols_bar = [C['neutral'], C['primary'], C['tertiary'], C['secondary'], C['highlight']]
bars = ax2.barh(methods, r2_vals, color=cols_bar, alpha=0.82, height=0.5)
ax2.axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
for bar, val in zip(bars, r2_vals):
    ax2.text(val+0.005, bar.get_y()+bar.get_height()/2,
             f'{val:.4f}', fontproperties=FP, fontsize=9, va='center')
ax2.set_xlabel('样本外 R²（OOS R2）', fontproperties=FP, fontsize=11)
ax2.set_title('五种方法样本外预测能力对比', fontproperties=FPB, fontsize=12)
for tick in ax2.get_yticklabels(): tick.set_fontproperties(FP)

fig.suptitle(f'惩罚回归案例分析（n={n}, p={p}, s={s}）',
             fontproperties=FPB, fontsize=13)
fig.savefig('figs/fig_B_case_comparison.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('图形已保存至 figs/fig_B_case_comparison.png')

---
## 提示词模板 #3：多方法样本外预测对比

```
背景：对高维数据做五种方法的样本外预测对比。

我的数据：X_train、y_train、X_test、y_test（均已完成标准化）
特征数 p 约 50，样本量 n 约 300。

请帮我：
1. 拟合 OLS、LassoCV(cv=10)、RidgeCV(cv=10)、ElasticNetCV(cv=10)、Post-Lasso
2. 汇总测试集的 MSE、RMSE、样本外 R2 到 DataFrame 并打印
3. 绘制横向条形图（OOS R2 对比，从低到高排序）
4. 打印 Lasso 和弹性网选中的变量数量
5. 所有代码用中文注释，random_state=42
```


In [ ]:
print('=' * 55)
print('Chapter B 案例分析完成')
print('=' * 55)
best_m = df_res['OOS_R2'].idxmax()
print(f'最优方法（OOS R2 最高）: {best_m}')
print(f'OOS R2 = {df_res.loc[best_m, "OOS_R2"]:.4f}')
print()
print('输出文件:')
print('  data/sim_B_sparse.csv')
print('  figs/fig_B_case_comparison.png')